In [1]:
import sys, platform, importlib.metadata, json
print("AUDIT_KERNEL="+json.dumps({"python":sys.version,"executable":sys.executable,"platform":platform.platform(),"packages":dict(sorted((d.metadata["Name"],d.version) for d in importlib.metadata.distributions()))}))

AUDIT_KERNEL={"python": "3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]", "executable": "/home/miakhlov/course-production-verified-20261118/scripts/environment/.venv-chem/bin/python", "platform": "Linux-6.18.58.1-microsoft-standard-WSL2-x86_64-with-glibc2.39", "packages": {"Jinja2": "3.1.6", "MarkupSafe": "3.0.4", "PyYAML": "6.0.3", "Pygments": "2.21.0", "Send2Trash": "2.2.0", "altair": "6.3.0", "anyio": "4.16.0", "argon2-cffi": "25.1.0", "argon2-cffi-bindings": "26.2.0", "arrow": "1.4.0", "asttokens": "3.0.2", "async-lru": "2.4.0", "attrs": "26.1.0", "babel": "2.18.0", "beautifulsoup4": "4.15.0", "bleach": "6.4.0", "certifi": "2026.10.23", "cffi": "2.1.1", "charset-normalizer": "3.5.2", "click": "8.5.1", "cloudpickle": "3.1.3", "comm": "0.2.3", "contourpy": "1.4.0", "cycler": "0.12.1", "debugpy": "1.8.22", "defusedxml": "0.7.1", "executing": "2.3.0", "fastjsonschema": "2.22.3", "fonttools": "4.68.0", "fqdn": "1.6.0", "h11": "0.16.0", "httpcore": "1.0.9", "httpx": "0.28.1", "idna": 

# Разбор A. Среда и растворитель

Предложение выбрать растворитель только по logP оставляет неизвестными pH и форму. Для нейтральной кислоты B растворимость и доля нейтральной формы могут меняться в противоположных направлениях. На линии A оставляем нейтральный базовый контроль, а численную pH-модель изучаем на условной одноосновной кислоте. Её pKa не является измерением аторвастатина.

Используйте явно условные pKa=4.5 и S0=10^-5 mol/L. Постройте долю нейтральной формы и общую растворимость в диапазоне pH 2–8. Проверьте pH=pKa аналитически и измените pKa на ±0.5. Эта чувствительность показывает неопределённость параметра, а не доверительный интервал измерения.

In [2]:
from pathlib import Path
import sys,json,math
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,compound,esol,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=120,bbox_inches='tight')
    display(fig)
    plt.close(fig)
OUT=workspace('M04-L02','case')


Сначала проверяем входы и единицы. Словари сохраняют названия полей вместе со значениями; список сохраняет соответствие записей. Выведите промежуточный результат перед дальнейшим анализом.

In [3]:
import numpy as np
pH=np.linspace(2,8,121);pka=4.5;s0=1e-5
neutral=1/(1+10**(pH-pka));solubility=s0*(1+10**(pH-pka))
print({'at_pKa':{'neutral_fraction':.5,'solubility_mol_L':2*s0},'parameters':'simulation'})


{'at_pKa': {'neutral_fraction': 0.5, 'solubility_mol_L': 2e-05}, 'parameters': 'simulation'}


Теперь сравним модель с контрольным случаем. `assert` останавливает расчёт при нарушении контракта; это защита от ошибочного продолжения, а не оценка научного понимания.

In [4]:
fig,axes=plt.subplots(1,2,figsize=(9,3))
for delta in [-.5,0,.5]:
    axes[0].plot(pH,1/(1+10**(pH-(pka+delta))),label=f'pKa={pka+delta}')
    axes[1].semilogy(pH,s0*(1+10**(pH-(pka+delta))))
axes[0].set(xlabel='pH',ylabel='Neutral fraction');axes[0].legend()
axes[1].set(xlabel='pH',ylabel='Simulated total S / mol L^-1');publish_plot(fig,'ph.png')


<Figure size 900x300 with 2 Axes>

Сохраним именно полученный результат. Не переносите этот вывод на другую химическую форму, мишень или экспериментальные условия без отдельной проверки.

In [5]:
result={'pKa_simulated':pka,'S0_simulated_mol_L':s0,'kind':'simulation','atorvastatin_parameter_measurement':'NOT_RUN'}


In [6]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M04-L02','A',result,ROOT,inputs=['practicuum/shared/data/compounds.json'])
print('Versioned evidence:',receipt)


Versioned evidence: /home/miakhlov/course-production-verified-20261118/practicuum/evidence/20261118-full-course-linux/workspace/my_research/evidence/M04-L02/20261118T141943451364Z-A-76f3b4b6949847d0a43f2346354118ce.json


Увеличение модельной растворимости не доказывает увеличение мембранного переноса. В B дополнительно нужны стабильность кислоты/лактона и состав среды. Дальше сравним структуры, чтобы выбирать информативные аналоги.

Перед переходом ответьте: какие результаты получены вычислением, какие заданы условно, и какая независимая проверка ещё отсутствует?